# Power Grid Alarm Classification with GNN


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp
import pandapower.networks as pn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report,confusion_matrix,ConfusionMatrixDisplay,precision_recall_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from xgboost import plot_importance

In [3]:
def sample_scenario(base_net, rng):
    net = base_net.deepcopy()
    for ld in net.load.index:
        net.load.at[ld,'p_mw']*=rng.uniform(0.7,1.3)
        net.load.at[ld,'q_mvar']*=rng.uniform(0.7,1.3)
    import numpy as np
    if rng.random()<0.5:
        out = rng.choice(net.line.index)
        net.line.at[out,'in_service']=False
    try:
        pp.runpp(net,enforce_q_lims=True,init='results')
    except:
        return None,None
    vm = net.res_bus.vm_pu.values
    p_load=np.zeros(len(net.bus))
    for _,r in net.load.iterrows():
        p_load[int(r.bus)]+=float(r.p_mw)
    voltage_alarm=((vm<0.95)|(vm>1.05)).astype(int)
    thermal_alarm_line=(net.res_line.loading_percent.values>100).astype(int)
    thermal_alarm_bus=np.zeros(len(net.bus),dtype=int)
    for idx,row in net.line.iterrows():
        if row.in_service and thermal_alarm_line[idx]:
            thermal_alarm_bus[int(row.from_bus)]=1
            thermal_alarm_bus[int(row.to_bus)]=1
    alarm_flag=voltage_alarm+2*thermal_alarm_bus
    bus_df=pd.DataFrame({'bus':net.bus.index,'voltage':vm,'load_MW':p_load,'alarm_flag':alarm_flag})
    active=net.line[net.line.in_service]
    edge_df=pd.DataFrame({'from_bus':active.from_bus.values,'to_bus':active.to_bus.values})
    return bus_df,edge_df

rng=np.random.default_rng(0)
base=pn.case14()
bus_list=[]
edge_list=[]
for i in range(300):
    b,e=sample_scenario(base,rng)
    if b is None: continue
    b['scenario']=i; e['scenario']=i
    bus_list.append(b); edge_list.append(e)
bus_all=pd.concat(bus_list,ignore_index=True)
edge_all=pd.concat(edge_list,ignore_index=True)
bus_all['alarm_binary']=(bus_all['alarm_flag']>0).astype(int)

/var/folders/0x/j3pcyrd14mg9tdfzmbqpkz_m0000gn/T/ipykernel_51451/1446353274.py:2: DeprecationWarning: Use copy.deepcopy(net) instead of net.deepcopy()
  net = base_net.deepcopy()
numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)
/var/folders/0x/j3pcyrd14mg9tdfzmbqpkz_m0000gn/T/ipykernel_51451/1446353274.py:2: DeprecationWarning: Use copy.deepcopy(net) instead of net.deepcopy()
  net = base_net.deepcopy()
numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)
/var/folders/0x/j3pcyrd14mg9tdfzmbqpkz_m0000gn/T/ipykernel_51451/1446353274.py:2: DeprecationWarning: Use copy.deepcopy(net) instead of net.deepcopy()
  net = base_net.deepcopy()
numba cannot be imp

In [4]:
def build_graph(bus_df,edge_df):
    bus_df=bus_df.copy(); edge_df=edge_df.copy()
    bus_df['bus']=bus_df['bus'].astype(str)
    edge_df['from_bus']=edge_df['from_bus'].astype(str)
    edge_df['to_bus']=edge_df['to_bus'].astype(str)
    mapping={b:i for i,b in enumerate(bus_df['bus'])}
    src=edge_df['from_bus'].map(mapping).to_numpy()
    dst=edge_df['to_bus'].map(mapping).to_numpy()
    edge_idx=np.vstack([src,dst])
    X=bus_df[['voltage','load_MW']].to_numpy(float)
    scaler=StandardScaler().fit(X)
    Xn=scaler.transform(X)
    y=bus_df['alarm_binary'].to_numpy(int)
    return edge_idx,Xn,y,scaler,mapping

edge_index,X,y,scaler,mapping = build_graph(bus_all,edge_all)

sss=StratifiedShuffleSplit(n_splits=1,train_size=0.7,random_state=42)
(tr,te),=sss.split(np.zeros_like(y),y)
Xtr,Xte=X[tr],X[te]
ytr,yte=y[tr],y[te]

In [5]:
clf=LogisticRegression(max_iter=2000,class_weight='balanced')
clf.fit(Xtr,ytr)
yp=clf.predict(Xte)
print('LR F1=',f1_score(yte,yp))

LR F1= 0.9336283185840708


In [6]:
device='cuda' if torch.cuda.is_available() else 'cpu'
def to_pyg(edge,X,y):
    return Data(
        x=torch.tensor(X,dtype=torch.float32,device=device),
        edge_index=torch.tensor(edge,dtype=torch.long,device=device),
        y=torch.tensor(y,dtype=torch.long,device=device)
    )

data=to_pyg(edge_index,X,y)
train_idx=torch.tensor(tr,dtype=torch.long,device=device)
test_idx=torch.tensor(te,dtype=torch.long,device=device)

class GCN(nn.Module):
    def __init__(self,in_dim,hidden=32):
        super().__init__()
        self.g1=GCNConv(in_dim,hidden)
        self.g2=GCNConv(hidden,hidden)
        self.fc=nn.Linear(hidden,2)
    def forward(self,x,edge):
        x=self.g1(x,edge); x=F.relu(x)
        x=self.g2(x,edge); x=F.relu(x)
        return self.fc(x)

model=GCN(2).to(device)
opt=torch.optim.Adam(model.parameters(),lr=1e-2,weight_decay=5e-4)

for epoch in range(120):
    model.train()
    opt.zero_grad()
    out=model(data.x,data.edge_index)
    loss=F.cross_entropy(out[train_idx],data.y[train_idx])
    loss.backward(); opt.step()
    if epoch%20==0: print(epoch,loss.item())

0 0.72666335105896
20 0.24490109086036682
40 0.12941622734069824
60 0.06115125119686127
80 0.03193061053752899
100 0.02067285031080246


In [7]:
model.eval()
with torch.no_grad():
    logits=model(data.x,data.edge_index)[test_idx]
    preds=logits.argmax(dim=-1).cpu().numpy()
print("GCN F1=",f1_score(yte,preds))


GCN F1= 0.9977595220313666
